# 🍷 Laboratorio: PCA con el caso de los vinos

## 📖 La historia
Un laboratorio enológico del **Piamonte (Italia)** analizó **178 vinos** de la misma región, elaborados con **tres cultivares** (variedades de uva) distintos. A cada vino le midieron **13 propiedades químicas**: alcohol, acidez, fenoles, color, prolina…

Cada análisis cuesta tiempo y reactivos. La directora del laboratorio sospecha que muchas de esas medidas **dicen lo mismo** y nos pregunta:

> ### ❓ ¿Cuántas medidas químicas necesitamos realmente para distinguir los vinos?

## 🧭 Lo que vas a aprender
1. Detectar **redundancia** entre variables con la matriz de correlación
2. Ver por qué PCA **sin escalar** da resultados engañosos
3. Elegir cuántos componentes conservar con el **scree plot**, la varianza acumulada y Kaiser
4. **Interpretar** componentes con cargas y biplot, en lenguaje del negocio
5. Usar PCA **antes de K-Means** y comparar resultados con una métrica común (ARI)

## ⏱️ Agenda (3 horas)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación | 10 min |
| 1 | Conocer los datos | 10 min |
| 2 | Correlaciones: ¿cuánta redundancia hay? | 15 min |
| 3 | PCA sin escalar: algo no cuadra | 20 min |
| 4 | Escalar y decidir cuántos componentes | 25 min |
| ☕ | Descanso | 10 min |
| 5 | Cargas: ponerle nombre a los componentes | 25 min |
| 6 | Biplot y cultivares | 15 min |
| 7 | K-Means sobre componentes vs sobre las 13 variables | 20 min |
| 8 | De componentes a medidas: ¿3 análisis en lugar de 13? | 15 min |
| 9 | Conclusiones | 15 min |

## 📋 Las variables
| Variable | Qué mide | Rango en los datos |
|---|---|---|
| `alcohol` | Grado alcohólico (% vol.) | 11.0 – 14.8 |
| `malic_acid` | Ácido málico (acidez "verde", de manzana) | 0.7 – 5.8 |
| `ash` | Cenizas: minerales que quedan al quemar el vino | 1.4 – 3.2 |
| `alcalinity_of_ash` | Alcalinidad de las cenizas | 10.6 – 30.0 |
| `magnesium` | Magnesio (mg/L) | 70 – 162 |
| `total_phenols` | Fenoles totales (taninos, pigmentos, aromas) | 1.0 – 3.9 |
| `flavanoids` | Flavonoides: la familia principal de fenoles | 0.3 – 5.1 |
| `nonflavanoid_phenols` | Fenoles que no son flavonoides | 0.1 – 0.7 |
| `proanthocyanins` | Proantocianidinas (taninos condensados: astringencia) | 0.4 – 3.6 |
| `color_intensity` | Intensidad del color | 1.3 – 13.0 |
| `hue` | Tono del color | 0.5 – 1.7 |
| `od280_od315` | Cociente de absorbancias OD280/OD315 (contenido de fenoles y proteínas) | 1.3 – 4.0 |
| `proline` | Prolina, un aminoácido (mg/L) | 278 – 1680 |
| `cultivar` | 🏷️ Cultivar (0, 1, 2). **Sólo para validar al final**: en un problema real no la tendríamos | — |

Fuente: *Wine*, UCI Machine Learning Repository (Forina et al.), incluido en `sklearn.datasets`.

---
# 0️⃣ Preparación (10 min)

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_metricas_k(tabla)` | Silhouette, Davies-Bouldin y Calinski-Harabasz según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_metricas_k(tabla):
    """Silhouette, Davies-Bouldin y Calinski-Harabasz para cada k (tabla de tabla_metricas_k)."""
    metricas = [
        ('silhouette', 'Silhouette (↑ mejor)'),
        ('davies_bouldin', 'Davies-Bouldin (↓ mejor)'),
        ('calinski_harabasz', 'Calinski-Harabasz (↑ mejor)'),
    ]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in metricas])
    for i, (col, _) in enumerate(metricas, start=1):
        fig.add_trace(go.Scatter(x=tabla['k'], y=tabla[col], mode='lines+markers', name=col), row=1, col=i)
        fig.update_xaxes(title_text='k', dtick=1, row=1, col=i)
    fig.update_layout(width=1150, height=400, showlegend=False, title='Métricas internas según k')
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido, silhouette, Davies-Bouldin y Calinski-Harabasz (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y métricas en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [ ]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Métricas internas de un agrupamiento. El ruido (-1) se excluye de las métricas,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
        fila['davies_bouldin'] = float(round(davies_bouldin_score(X[mask], labels[mask]), 3))
        fila['calinski_harabasz'] = float(round(calinski_harabasz_score(X[mask], labels[mask]), 1))
    else:
        fila.update({'silhouette': np.nan, 'davies_bouldin': np.nan, 'calinski_harabasz': np.nan})
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con inercia, silhouette, Davies-Bouldin y Calinski-Harabasz."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), **{
            m: fila[m] for m in ('silhouette', 'davies_bouldin', 'calinski_harabasz')}})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Cargar los datos

`load_wine` viene con scikit-learn: no hay que descargar nada. Renombramos dos columnas para que sean más cómodas de escribir.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
df = wine.frame.rename(columns={
    'od280/od315_of_diluted_wines': 'od280_od315',
    'target': 'cultivar'
})

columnas = [
    'alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
    'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins',
    'color_intensity', 'hue', 'od280_od315', 'proline'
]
df.head()

---
# 1️⃣ Conocer los datos (10 min)

Antes de reducir dimensiones hay que saber **qué tenemos**: cuántos vinos, si faltan datos y en qué escala está cada medida.

In [ ]:
# 1. Imprime df.shape
# 2. Imprime el total de nulos: df.isnull().sum().sum()
# 3. Muestra las estadísticas de las 13 medidas: df[columnas].describe().round(2).T
# ✍️ Tu código aquí


In [ ]:
# ¿Cuántos vinos hay de cada cultivar?
df['cultivar'].value_counts().sort_index()

✍️ **¿Qué observas?**
- ¿Cuántos vinos y cuántas medidas hay? ¿Hay nulos?
- Compara el rango de `proline` con el de `hue`. ¿Están en escalas parecidas?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ Correlaciones: ¿cuánta redundancia hay? (15 min)

Si dos medidas están muy correlacionadas, medir las dos aporta poco: una casi predice a la otra. PCA aprovecha justamente esa **redundancia**.

In [ ]:
# Matriz de correlación de las 13 medidas
# Pista: plot_correlation_matrix(df, columnas)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Pares de medidas más correlacionadas (en valor absoluto)
corr = df[columnas].corr()
pares = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    .stack()
    .dropna()
    .rename('correlacion')
    .reset_index()
    .rename(columns={'level_0': 'medida_1', 'level_1': 'medida_2'})
)
pares = pares.reindex(pares['correlacion'].abs().sort_values(ascending=False).index)
print(f"Pares con |r| > 0.5: {(pares['correlacion'].abs() > 0.5).sum()} de {len(pares)}")
pares.head(8).round(2)

✍️ **¿Qué observas?**
- ¿Qué medidas están más correlacionadas entre sí? ¿Ves "familias" de variables?
- ¿Cuántos pares tienen correlación mayor a 0.5? ¿Qué te dice eso sobre la redundancia?

RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ PCA sin escalar: algo no cuadra (20 min)

Primero hagamos lo "obvio": PCA directo sobre las 13 medidas, tal como vienen. `PCA()` sin `n_components` calcula **todos** los componentes (13).

In [ ]:
from sklearn.decomposition import PCA

# 1. pca_crudo = PCA().fit(df[columnas])      ← sin escalar
# 2. plot_varianza_explicada(pca_crudo)
# 3. Imprime la varianza explicada por PC1: pca_crudo.explained_variance_ratio_[0]
# ✍️ Tu código aquí


🤔 ¿Cuánta varianza explica PC1? ¿Te parece creíble que 13 medidas químicas se resuman en un solo número? Veamos de qué está hecho PC1.

In [ ]:
# Tabla con dos columnas, una fila por medida:
# 1. 'carga_PC1': pca_crudo.components_[0]
# 2. 'pct_varianza_total': 100 * df[columnas].var() / df[columnas].var().sum()
# Pista: pd.DataFrame({...}, index=columnas).sort_values('pct_varianza_total', ascending=False).round(3)
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Qué medida domina PC1? ¿Cuánto vale su carga?
- ¿Qué porcentaje de la varianza total aporta esa medida? ¿Por qué crees que pasa?
- ¿PCA encontró un patrón químico o algo más trivial?

RELLENAR CON TUS COMENTARIOS

---
# 4️⃣ Escalar y decidir cuántos componentes (25 min)

Con `StandardScaler` cada medida queda con media 0 y desviación estándar 1: todas compiten en igualdad de condiciones. Después ajustamos un PCA con los 13 componentes para ver el scree plot completo.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. scaler = StandardScaler()
# 2. X_scaled = scaler.fit_transform(df[columnas])
# 3. pca_vinos = PCA().fit(X_scaled)
# 4. plot_varianza_explicada(pca_vinos)
# ✍️ Tu código aquí


Ahora aplicamos los tres criterios de fundamentos: **varianza acumulada** (80% y 95%), **Kaiser** (eigenvalor > 1) y el **codo** del scree plot.

In [ ]:
# 1. tabla_varianza = pd.DataFrame con columnas:
#      'componente' (PC1, PC2, ...), 'eigenvalor' (pca_vinos.explained_variance_),
#      'varianza' (pca_vinos.explained_variance_ratio_) y 'acumulada' (np.cumsum de la anterior)
# 2. ¿Cuántos componentes para 80% y 95%?  Pista: np.argmax(acumulada >= 0.80) + 1
# 3. Kaiser: (pca_vinos.explained_variance_ > 1).sum()
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Cuánta varianza explican PC1, PC2 y PC3 ahora? Compáralo con la Parte 3
- ¿Cuántos componentes pide cada criterio (80%, 95%, Kaiser, codo)? ¿Coinciden?
- ¿Cuántos usarías para **visualizar** y cuántos para **comprimir**?

RELLENAR CON TUS COMENTARIOS

---
# ☕ Descanso (10 min)

---
# 5️⃣ Cargas: ponerle nombre a los componentes (25 min)

Un componente es una **mezcla** de las 13 medidas. Para que sirva al laboratorio hay que **traducirlo a lenguaje de enólogo**: ¿qué tipo de vino tiene un PC1 alto?

In [ ]:
# Mapa de calor de las cargas de los primeros 3 componentes
# Pista: plot_cargas(pca_vinos, columnas, n_componentes=3)
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Las medidas con más peso (en valor absoluto) en PC1 y PC2, con su signo
cargas = pd.DataFrame(
    pca_vinos.components_[:3].T,
    index=columnas,
    columns=['PC1', 'PC2', 'PC3']
)
for pc in ['PC1', 'PC2']:
    top = cargas[pc].reindex(cargas[pc].abs().sort_values(ascending=False).index).head(6)
    print(f"--- {pc} ---")
    print(top.round(2).to_string())
    print()

✍️ **Ponle nombre a los componentes.** Completa la tabla con las medidas de mayor carga y un nombre que entendería un enólogo:

| Componente | Suben con él (+) | Bajan con él (−) | Nombre de enólogo |
|---|---|---|---|
| **PC1** (____%) | ______ | ______ | ______ |
| **PC2** (____%) | ______ | ______ | ______ |

¿Qué familias de variables de la Parte 2 reconoces en cada componente?

RELLENAR CON TUS COMENTARIOS

---
# 6️⃣ Biplot y cultivares (15 min)

Proyectamos los 178 vinos sobre PC1 y PC2 (`transform`) y dibujamos el biplot. Primero **sin colores**: así lo vería un analista que no conoce los cultivares.

In [ ]:
# 1. scores_vinos = pca_vinos.transform(X_scaled)
# 2. plot_biplot(scores_vinos, pca_vinos, columnas, title='Biplot de los vinos (sin etiquetas)')
# ✍️ Tu código aquí


¿Ves grupos? Ahora usamos el cultivar **sólo para validar** lo que PCA encontró sin conocerlo.

⚠️ En la práctica **no tendríamos** esta etiqueta: si la tuviéramos, el problema sería de clasificación (supervisado).

In [ ]:
# Mismo biplot, coloreado por cultivar
# 1. etiqueta_cultivar = 'Cultivar ' + df['cultivar'].astype(str)
# 2. plot_biplot(scores_vinos, pca_vinos, columnas, labels=etiqueta_cultivar, title='Biplot coloreado por cultivar')
# ✍️ Tu código aquí


In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
# Posición promedio de cada cultivar en PC1-PC2 y algunas medidas originales
resumen_cultivar = pd.DataFrame(scores_vinos[:, :2], columns=['PC1', 'PC2']).groupby(df['cultivar']).mean()
resumen_cultivar = resumen_cultivar.join(
    df.groupby('cultivar')[['flavanoids', 'alcohol', 'color_intensity', 'proline']].mean()
)
resumen_cultivar.round(2)

✍️ **¿Qué observas?**
- Sin colores, ¿cuántos grupos intuyes?
- Usando las flechas y la tabla: ¿cómo describirías a cada cultivar (fenoles, alcohol, color)?
- ¿Qué cultivares se traslapan más?

RELLENAR CON TUS COMENTARIOS

---
# 7️⃣ K-Means sobre componentes vs sobre las 13 variables (20 min)

Ahora sí respondemos con evidencia: agrupamos los vinos con K-Means (**k = 3**) en distintos espacios y medimos qué tan bien recupera los cultivares con el **ARI** (*Adjusted Rand Index*: 1 = coincidencia perfecta, 0 = como al azar).

⚠️ **Cómo comparar con justicia.** El silhouette de un modelo en 2 dimensiones y el de otro en 13 **no** están en la misma escala: en menos dimensiones las distancias son distintas y el silhouette suele salir más alto "gratis". Por eso:
- Cada K-Means se **ajusta** en su propio espacio (13 variables, PC1-PC2, los componentes del 80%)
- Pero **todas** las métricas internas se calculan sobre la misma referencia: `X_scaled` (las 13 medidas escaladas)
- Y la métrica principal es el **ARI**, que sólo compara etiquetas y no depende del espacio

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

comparacion = []


def registrar(nombre, labels):
    """Métricas internas sobre la MISMA referencia (X_scaled) + ARI contra el cultivar."""
    fila = evaluar_clustering(nombre, X_scaled, labels)
    fila['ARI_vs_cultivar'] = round(adjusted_rand_score(df['cultivar'], labels), 3)
    comparacion.append(fila)
    return pd.DataFrame([fila])


# Línea base 1: las 13 medidas SIN escalar (el error de la Parte 3)
kmeans_crudo = KMeans(n_clusters=3, n_init=10, random_state=42).fit(df[columnas])
registrar('13 medidas sin escalar', kmeans_crudo.labels_)

# Línea base 2: las 13 medidas escaladas
kmeans_13 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_scaled)
registrar('13 medidas escaladas', kmeans_13.labels_)

In [ ]:
# 1. kmeans_pc2 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(scores_vinos[:, :2])
# 2. registrar('PC1-PC2', kmeans_pc2.labels_)
# 3. n_80 = número de componentes para el 80% (lo calculaste en la Parte 4)
# 4. Repite con scores_vinos[:, :n_80] → kmeans_pc80; nombre f'{n_80} PCs (80% var.)'
# 5. pd.DataFrame(comparacion)
# ✍️ Tu código aquí


In [ ]:
# Tabla cruzada cultivar vs cluster de kmeans_pc2
# Pista: pd.crosstab(df['cultivar'], kmeans_pc2.labels_, rownames=['cultivar'], colnames=['cluster'])
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Qué ARI obtiene cada modelo? ¿Cuánto se pierde al pasar de 13 medidas a 2 componentes?
- ¿Para qué sirvió escalar?
- En la tabla cruzada, ¿qué cultivar se confunde más?
- ¿Por qué no comparamos silhouettes calculados en espacios distintos?

RELLENAR CON TUS COMENTARIOS

---
# 8️⃣ De componentes a medidas: ¿3 análisis en lugar de 13? (15 min)

🤔 Hay una trampa en la conclusión anterior: para calcular PC1 y PC2 de un vino nuevo **necesitamos sus 13 medidas** (cada componente es una mezcla de todas). PCA reduce **columnas**, no **análisis de laboratorio**.

Si la directora quiere **medir menos**, las cargas nos dicen qué medir: escogemos una medida representativa de cada "familia":
- La de mayor carga en **PC1**
- La de mayor carga en **PC2**
- Una que separe al cultivar que esas dos distinguen peor (mira el biplot y la tabla de la Parte 6)

In [ ]:
# 1. medidas_clave = [ ... ]   ← elige 3 medidas siguiendo las reglas de arriba
# 2. scaler_clave = StandardScaler(); X_clave = scaler_clave.fit_transform(df[medidas_clave])
# 3. kmeans_clave = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_clave)
# 4. registrar('3 medidas clave', kmeans_clave.labels_)
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Qué 3 medidas elegiste y por qué?
- ¿Qué ARI obtuviste? ¿Cuánto pierdes frente a las 13 medidas? ¿Vale la pena el ahorro?

RELLENAR CON TUS COMENTARIOS

---
# 9️⃣ Conclusiones (15 min)

### 📊 Tabla comparativa
Todas las métricas internas están calculadas sobre la misma referencia (`X_scaled`); el ARI compara cada agrupamiento con el cultivar real.

In [ ]:
pd.DataFrame(comparacion)

## ❓ Respuesta a la pregunta
> *¿Cuántas medidas químicas necesitamos realmente para distinguir los vinos?*

✍️ Responde con tus resultados: ¿cuántas dimensiones tiene realmente la química de estos vinos y cómo las llamarías? ¿Cuántas **medidas** recomendarías al laboratorio y con qué costo en ARI?

RELLENAR CON TUS COMENTARIOS

## 📝 Lo que aprendimos
✍️ Escribe 3 cosas que aprendiste hoy.

RELLENAR CON TUS COMENTARIOS

## 🚀 Retos opcionales
- Cambia la tercera medida clave por `alcohol`. ¿Sube o baja el ARI? ¿Por qué crees que pasa?
- Ajusta K-Means con varios k sobre PC1-PC2 con `tabla_metricas_k(scores_vinos[:, :2], range(2, 7))`. ¿Las métricas internas eligen el mismo número de grupos que cultivares hay, sin conocerlos?
- Repite la Parte 7 usando sólo PC1. ¿Cuánto ARI pierdes?